<center>
  <a>
    <img src="https://weclouddata.s3.amazonaws.com/images/logos/wcd_logo_new_2.png" style="zoom: 50%;">
    <div style="height: 20px;"></div> <!-- Empty div for spacing -->
    <h1 </h1>
  </a>
</center>

# PEFT Demo

Large Language Models (LLMs) are powerful, but they are trained to be general-purpose. In real applications—such as chatbots, customer support, or internal knowledge systems , we need models that understand our domain, tone, and data.

Fully fine-tuning an LLM is expensive and slow because it requires updating billions of parameters.
PEFT (Parameter-Efficient Fine-Tuning) solves this by freezing the base model and training only a small set of lightweight adapter weights. This makes customization fast, affordable, and practical.

In this notebook, we will use a PEFT method (such as LoRA) to adapt a pretrained model and observe how it becomes better aligned with our target task—without retraining the entire model.

In [1]:
# Install dependencies
!pip install -q transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.0 MB/s eta 0:00:00


In [8]:
import sys
!{sys.executable} -m pip install -q "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 35.9 MB/s eta 0:00:00


In [2]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, TaskType

In [3]:
# CREATE INSTRUCTION DATASET
# Focused on Q&A about general knowledge - easier for model to learn
# More examples, consistent format, better quality
data = {
    "instruction": [
        # Basic bootcamp information
        "What is the bootcamp about?",
        "Where is the bootcamp located?",
        "When did Week 0 sessions take place?",
        "What time do the daily sessions start?",
        "What time do the daily sessions end?",
        "What is the timezone for the bootcamp?",
        "Is attendance mandatory?",
        "What is the Zoom link for the bootcamp?",

        # Week 0 activities
        "What surveys were students required to complete in Week 0?",
        "What is the purpose of the student background survey?",
        "What does the AI skills and goals survey assess?",
        "Where can students submit questions or issues?",
        "What was the class schedule for Week 0?",

        # Project-related questions
        "What is the capstone project about?",
        "When did students start working on the capstone project?",
        "What are some project ideas mentioned?",
        "How many groups were formed for projects?",
        "When is the project proposal due?",
        "What should be included in the project proposal?",
        "How are project proposals submitted?",

        # Project ideas
        "What are the AI for Education project focuses?",
        "What is the AI for RFP project about?",
        "What is the AI for HR project about?",
        "What is the Enterprise AI Hub project?",
        "What is the Apache Doris/VeloDB project?",

        # Technical topics covered
        "What Python topics were covered in Week 1?",
        "What is covered in the Pandas module?",
        "What machine learning topics were taught?",
        "What NLP topics were introduced?",
        "What library is used for sentence embeddings?",

        # Instructors and support
        "Who are the instructors mentioned?",
        "How can students get help during the bootcamp?",
        "Where can students find office hours?",
        "How do students provide instructor feedback?",

        # Schedule and holidays
        "What holiday occurred during the bootcamp?",
        "When was Saudi National Day?",
        "What week did the bootcamp officially start the capstone?",

        # Technical tools and platforms
        "What platform is used for learning materials?",
        "What tools are used for collaboration?",
        "What is the Miro board used for?",
        "Where can students find open data for Saudi Arabia?",

        # Code examples
        "How do you flatten a nested list in Python?",
        "How do you use OpenAI's API in Python?",
        "What is the format for chat history in OpenAI?",
        "How do you create embeddings with sentence transformers?",

        # Project validation
        "How should teams conduct user validation?",
        "When should user validation surveys be created?",
        "Where do teams submit user validation links?",

        # Presentation requirements
        "When is the project proposal presentation?",
        "Is the first presentation graded?",
        "How many presentations are expected during the program?",

        # Model information
        "What is Claude Sonnet 4.5?",
        "What Claude models are available?",
        "How can you access Claude?",

        # Additional resources
        "Where can students find the LLM from scratch book?",
        "What is Modal and what is it used for?",
        "Where can students find the Google ROI of AI report?",

        # Client projects
        "What clients are available for the capstone project?",
        "When do client projects officially start?",
        "Can students change their project team?",
    ],
    "output": [
        # Basic bootcamp information
        "This is a GenAI MLOps/LLMOps Bootcamp organized by WeCloudData and SDA (Saudi Digital Academy) in Riyadh.",
        "The bootcamp is located in Riyadh, Saudi Arabia.",
        "Week 0 sessions took place from Monday, August 25 to Thursday, August 28, 2025.",
        "Daily sessions start at 9:00 AM Riyadh time.",
        "Daily sessions end at 12:00 PM Riyadh time (with some afternoon sessions extending to 5:00 PM).",
        "The bootcamp operates in Riyadh Time timezone.",
        "Yes, attendance is mandatory and is taken daily.",
        "The main Zoom link is https://us02web.zoom.us/j/83424596656 with passcode 726039.",

        # Week 0 activities
        "Students were required to complete two surveys: a student background survey and an AI skills and goals survey.",
        "The student background survey helps understand students' background, motivations, and technical familiarity to tailor the bootcamp experience.",
        "The AI skills and goals survey assesses students' current AI skills and learning objectives.",
        "Students can submit questions or issues in a shared Google spreadsheet for office hour tickets.",
        "Week 0 sessions ran Monday through Thursday (Aug 25-28, 2025) from 9:00 AM to 12:00 PM.",

        # Project-related questions
        "The capstone project involves developing AI solutions in areas like Education, RFP, HR, or Enterprise AI, working in teams to solve real-world problems.",
        "Students started researching capstone projects on September 1, 2025 (Week 1).",
        "Project ideas include: AI for Education (instructors, students, schools), AI for Sales/RFP, AI for HR (workforce planning, hiring, job applications), Enterprise AI Hub, Apache Doris/VeloDB QA chatbot, and Text to SQL for real-time data warehouse.",
        "There were 10 project groups formed (Group 1 through Group 10).",
        "The draft project proposal and presentation were due at the end of Week 2, with presentations on Thursday of Week 2.",
        "Proposals should include: pain points, market research, user research via class survey, proposed solution (AI product description and implementation plan), and can be in Word doc or presentation format.",
        "Project proposals are submitted through a Google Form, with teams also maintaining collaborative Google Docs.",

        # Project ideas
        "AI for Education focuses on creating solutions for instructors, students, and schools to improve learning experiences.",
        "AI for RFP (Request for Proposal) aims to automate and streamline the RFP response process using AI.",
        "AI for HR includes workforce planning, hiring/recruiting, and job application assistance.",
        "Enterprise AI Hub focuses on secure deployment of LLM, Chat, Knowledge Search, and Workflows for enterprises.",
        "Apache Doris/VeloDB project involves creating a QA chatbot and Text to SQL functionality for real-time data warehouses.",

        # Technical topics covered
        "Week 1 covered Python fundamentals including functions, nested lists, list comprehension, and recursion.",
        "The Pandas module covers data manipulation, DataFrames, SQL-style joins, and data analysis techniques.",
        "Machine learning topics include regression, ensemble trees (bagging), hyperparameter tuning, and credit risk modeling.",
        "NLP topics include text preprocessing, tokenization, stemming, TF-IDF vectorization, sentiment analysis, and sentence embeddings.",
        "SentenceTransformer library (e.g., 'all-MiniLM-L6-v2' model) is used for creating sentence embeddings.",

        # Instructors and support
        "Instructors mentioned include WCD-Idris, shaohua-weclouddata, WCD-Carlo, Esteban (Gen AI - WCD), and Prabesh Lamichhane.",
        "Students can get help by using the office hour tickets spreadsheet, asking in class, or through breakout rooms during project sessions.",
        "Office hours can be accessed through the Week 0 Office Hour Tickets Google spreadsheet.",
        "Students provide instructor feedback through anonymous Google Forms surveys, conducted multiple times during the bootcamp.",

        # Schedule and holidays
        "Tuesday, September 23rd, 2025 was a holiday for Saudi National Day.",
        "Saudi National Day was celebrated on September 23rd, 2025.",
        "The capstone project officially starts in Week 9 or Week 10, though preparation began in Week 1.",

        # Technical tools and platforms
        "The learning portal at sda.weclouddata.com is used for accessing course materials, videos, and assignments.",
        "Tools used include Google Docs for collaboration, Miro boards for group ideation, Google Forms for surveys, and Zoom for virtual sessions.",
        "The Miro board is used for group assignment, project ideation, and organizing team information.",
        "Open data for Saudi Arabia can be found at https://data.gov.sa/ar, the National Data Bank portal.",

        # Code examples
        "You can flatten a nested list using recursion: check if each element is a list using isinstance(), recursively call the function on list elements, and append non-list elements to the result.",
        "To use OpenAI's API: import openai, create a client with openai.OpenAI(api_key=api_key), then call client.chat.completions.create() with model and messages parameters.",
        "Chat history format is a list of dictionaries with 'role' and 'content' keys: [{'role': 'user', 'content': 'question'}, {'role': 'assistant', 'content': 'answer'}].",
        "To create embeddings: from sentence_transformers import SentenceTransformer, load model with SentenceTransformer('all-MiniLM-L6-v2'), then call model.encode(sentences).",

        # Project validation
        "Teams should create user validation surveys, have classmates complete them, and track responses to validate their project ideas.",
        "User validation surveys should be created on Sunday (Week 2, Day 7) and distributed on Monday for completion.",
        "Teams submit and track user validation links in a shared Google spreadsheet.",

        # Presentation requirements
        "The project proposal presentation was held on Thursday of Week 2 (September 11, 2025).",
        "No, the first presentation in Week 2 was a practice presentation and was not graded.",
        "Students are expected to do 2-3 more presentations throughout the program, which will be graded.",

        # Model information
        "Claude Sonnet 4.5 is the smartest model in the Claude 4 family, efficient for everyday use, created by Anthropic.",
        "The Claude 4 family consists of Claude Opus 4.1, Claude Opus 4, Claude Sonnet 4.5, and Claude Sonnet 4.",
        "Claude can be accessed via web-based chat interface, mobile and desktop apps, API and developer platform (model string 'claude-sonnet-4-5-20250929'), and Claude Code command line tool.",

        # Additional resources
        "The book 'Build a Large Language Model (From Scratch)' by Sebastian Raschka is available as a shared PDF resource.",
        "Modal (modal.com) is a platform that provides free GPU access (smaller GPUs) for high-performance AI infrastructure and serverless computing.",
        "The Google Cloud ROI of AI 2025 report is available as a shared resource, emphasizing thinking about problems rather than just ideas.",

        # Client projects
        "Available clients include: WeCloudData (for Edtech AI projects), Beamdata (for MLOps/LLMOps projects), and VeloDB (starting Week 9).",
        "Client projects officially start in Week 9 or Week 10, though preparation and research begin earlier.",
        "Yes, students don't have to stay in the same project team for their final capstone project. Re-grouping happens when the real client project starts after Week 7.",
    ]
}

dataset = Dataset.from_dict(data)

In [4]:
# Split into train/test (80/20 split)
dataset = dataset.train_test_split(test_size=0.2, seed=42)

print(f"Training samples: {len(dataset['train'])}")
print(f"Test samples: {len(dataset['test'])}")
print(f"\nExample:\nQ: {dataset['train'][0]['instruction']}\nA: {dataset['train'][0]['output']}\n")

Training samples: 48
Test samples: 12

Example:
Q: What clients are available for the capstone project?
A: Available clients include: WeCloudData (for Edtech AI projects), Beamdata (for MLOps/LLMOps projects), and VeloDB (starting Week 9).



In [5]:
# LOAD MODEL AND TOKENIZER (Qwen2.5 - Modern SLM!)
model_name = "Qwen/Qwen2.5-0.5B-Instruct"  # You can also try "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.pad_token_id

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [6]:

# FORMAT DATA AS INSTRUCTION-RESPONSE PAIRS


def format_instruction(sample):
    """Format data in instruction-following format"""
    instruction = sample["instruction"]
    output = sample["output"]

    # Simple, clean format
    prompt = f"Question: {instruction}\nAnswer: {output}"

    return {"text": prompt + tokenizer.eos_token}

# Apply formatting
dataset = dataset.map(format_instruction, remove_columns=dataset["train"].column_names)

# Tokenize
def tokenize(sample):
    result = tokenizer(
        sample["text"],
        truncation=True,
        max_length=128,  # Shorter since our examples are concise
        padding=False
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize, remove_columns=["text"], batched=False)


Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

In [9]:

# CONFIGURE LoRA PEFT

lora_config = LoraConfig(
    r=8,                    # Rank of the low-rank matrices
    lora_alpha=16,          # Scaling factor
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Qwen2.5 attention modules
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Apply LoRA to model
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # Shows only ~0.3% parameters are trainable!


trainable params: 1,081,344 || all params: 495,114,112 || trainable%: 0.2184


In [10]:
# TRAINING CONFIGURATION


training_args = TrainingArguments(
    output_dir="./instruction_tuned_qwen0.5",
    num_train_epochs=10,  # More epochs since we have more data
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="epoch",  # Changed from evaluation_strategy for newer transformers
    save_strategy="epoch",
    load_best_model_at_end=True,
    warmup_steps=20,
    report_to="none",
    weight_decay=0.01,
)

# Custom data collator to ensure proper padding
from dataclasses import dataclass
from typing import Any, Dict, List

@dataclass
class DataCollatorForCompletionOnlyLM:
    tokenizer: Any

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        # Extract input_ids and labels
        input_ids = [f["input_ids"] for f in features]
        labels = [f["labels"] for f in features]

        # Find max length in batch
        max_length = max(len(ids) for ids in input_ids)

        # Pad sequences
        padded_input_ids = []
        padded_labels = []
        attention_mask = []

        for ids, lbls in zip(input_ids, labels):
            padding_length = max_length - len(ids)

            # Pad input_ids and attention_mask
            padded_input_ids.append(ids + [self.tokenizer.pad_token_id] * padding_length)
            attention_mask.append([1] * len(ids) + [0] * padding_length)

            # Pad labels with -100 (ignored in loss calculation)
            padded_labels.append(lbls + [-100] * padding_length)

        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long)
        }

data_collator = DataCollatorForCompletionOnlyLM(tokenizer=tokenizer)



In [11]:

# TRAIN THE MODEL

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
)

print("\n Starting training...\n")
trainer.train()



 Starting training...



Epoch,Training Loss,Validation Loss
1,No log,3.202940
2,3.259922,2.976579
3,3.259922,2.625601
4,2.730798,2.397582
5,2.141553,2.322544
6,2.141553,2.333104
7,1.762845,2.332800
8,1.762845,2.366899
9,1.560456,2.405890
10,1.350248,2.409873


TrainOutput(global_step=60, training_loss=2.134303585688273, metrics={'train_runtime': 37.8422, 'train_samples_per_second': 12.684, 'train_steps_per_second': 1.586, 'total_flos': 52123817011200.0, 'train_loss': 2.134303585688273, 'epoch': 10.0})

In [13]:

# TEST THE FINE-TUNED MODEL

def generate_response(question):
    """Generate a response for a given question"""
    prompt = f"Question: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=50,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract just the answer part
    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()
    return response

print("\n" + "="*60)
print(" TESTING THE FINE-TUNED MODEL")
print("="*60 + "\n")

# Test with examples from training distribution

test_cases = [
    # Direct matches from training (5 cases)
    "What is the bootcamp about?",  # From training
    "Where is the bootcamp located?",  # From training
    "Is attendance mandatory?",  # From training
    "What are some project ideas mentioned?",  # From training
    "What Python topics were covered in Week 1?",  # From training

    # Similar patterns to training (10 cases)
    "What time does the bootcamp start each day?",  # Similar to "What time do sessions start?"
    "How many surveys did students need to complete?",  # Similar to survey questions
    "Who can students contact for help?",  # Similar to support questions
    "When do students present their proposals?",  # Similar to presentation timing
    "What is the purpose of the capstone project?",  # Similar to project questions
    "Which tools are used for team collaboration?",  # Similar to tools questions
    "What happened on September 23rd, 2025?",  # Similar to holiday questions
    "How do you create a nested list in Python?",  # Similar to code questions
    "What machine learning algorithms were taught?",  # Similar to technical topics
    "Can students switch project teams later?",  # Similar to team questions

    # New but related to bootcamp domain (5 cases)
    "What programming languages are taught in the bootcamp?",  # New inference
    "How long is the entire bootcamp program?",  # Requires inference from weeks mentioned
    "What should I do if I'm sick and can't attend?",  # Combines sick leave info
    "Are there any breaks during the day?",  # Prayer break mentioned
    "What happens after Week 7 of the bootcamp?",  # Requires timeline understanding
]

for question in test_cases:
    print(f" Question: {question}")
    print(f" Answer: {generate_response(question)}")
    print("-" * 60)



 TESTING THE FINE-TUNED MODEL

 Question: What is the bootcamp about?
 Answer: The Bootcamp focuses on building a project management platform for large companies.
------------------------------------------------------------
 Question: Where is the bootcamp located?
 Answer: The Bootcamp is located in San Antonio, Texas.
------------------------------------------------------------
 Question: Is attendance mandatory?
 Answer: Yes, all students are expected to attend classes on time.
------------------------------------------------------------
 Question: What are some project ideas mentioned?
 Answer: Project idea 1: Create a virtual reality scavenger hunt for students.
------------------------------------------------------------
 Question: What Python topics were covered in Week 1?
 Answer: Week 1 covered object-oriented programming, data structures (list and dictionary), control flow (if-else statements), loops (for loop with range function), functions, classes, objects, files and data

In [14]:

# SAVE THE MODEL


model.save_pretrained("./instruction_tuned_qwen0.5")
tokenizer.save_pretrained("./instruction_tuned_qwen0.5")

print("\n Model saved! You can load it later with:")
print("   from peft import PeftModel, AutoPeftModelForCausalLM")
print("   model = AutoPeftModelForCausalLM.from_pretrained('./instruction_tuned_gpt2_final')")



 Model saved! You can load it later with:
   from peft import PeftModel, AutoPeftModelForCausalLM
   model = AutoPeftModelForCausalLM.from_pretrained('./instruction_tuned_gpt2_final')


In [15]:

# (OPTIONAL) COMPARE WITH BASE MODEL

print("\n" + "="*60)
print("COMPARISON: Base Model vs Fine-tuned")
print("="*60 + "\n")

# Load base model for comparison
base_model = AutoModelForCausalLM.from_pretrained(model_name)
base_model.eval()

test_question = "What is the bootcamp about?"
prompt = f"Question: {test_question}\nAnswer:"

print(f" Question: {test_question}\n")

# Base model response
inputs = tokenizer(prompt, return_tensors="pt")
with torch.no_grad():
    base_output = base_model.generate(
        **inputs,
        max_new_tokens=50,
        pad_token_id=tokenizer.pad_token_id,
        temperature=0.7,
        do_sample=True
    )
base_response = tokenizer.decode(base_output[0], skip_special_tokens=True)

print(f" Base Qwen2.5 Response:\n{base_response}\n")
print(f" Fine-tuned Model Response:\n{generate_response(test_question)}\n")

print("="*60)
print(" Demo complete!")
print("   The fine-tuned model should give focused, direct answers")
print("   while the base model may be more verbose or unfocused.")
print("="*60)


COMPARISON: Base Model vs Fine-tuned



Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

 Question: What is the bootcamp about?

 Base Qwen2.5 Response:
Question: What is the bootcamp about?
Answer: The Bootcamp was a 10-day course offered by Microsoft to help people learn programming languages and software development. It was created in 2008, with the aim of providing a practical, hands-on experience for students who are interested in

 Fine-tuned Model Response:
The bootcamp covers 4 topics - Python, AI, Machine Learning, and Data Science.

 Demo complete!
   The fine-tuned model should give focused, direct answers
   while the base model may be more verbose or unfocused.
